# 🚀 AI Agents Tutorial: Hands-On with LangChain & LangSmith

---

Welcome! This notebook walks through building and evaluating AI agents, step by step.

**What you'll learn:**
- How LLMs work
- What are agents
- Adding tools
- Memory & state
- Tracing with LangSmith
- Evaluating agents

**Time:** ~20 minutes

---

## 📦 Step 1: Setup

In [ ]:
# Install required packages
!pip install -q openai langchain langchain-core langchain-openai langgraph langsmith

import os
import getpass

# Set up API keys
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("🔑 Enter OpenAI API Key: ")

# Optional: LangSmith for tracing (get free key at https://smith.langchain.com)
if "LANGCHAIN_API_KEY" not in os.environ:
    use_langsmith = input("🕵️ Use LangSmith for tracing? (y/n): ").lower().strip() == "y"
    if use_langsmith:
        os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("🔑 Enter LangSmith API Key: ")
        os.environ["LANGCHAIN_TRACING_V2"] = "true"
        os.environ["LANGCHAIN_PROJECT"] = "agents-tutorial"
else:
    print("⏭️  Skipping LangSmith (traces will still work locally)")

print("✅ Setup complete!")

---

## 🤖 Step 2: Meet the LLM

In [ ]:
from openai import OpenAI

# Initialize the LLM
client = OpenAI()

# Simple question
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What is 2+2?"}],
    max_tokens=50
)

print("📍 The LLM says:", response.choices[0].message.content)

**What's happening:**
- We send a prompt to the LLM
- It generates a response
- Simple, but stateless - no memory!

---

## 🔧 Step 3: Adding Tools

In [ ]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

# Define a custom tool - calculator
@tool
def calculator(expression: str) -> str:
    """Calculate a math expression.
    
    Args:
        expression: A math expression like '2+2' or '25*17'
    
    Returns:
        The result of the calculation
    """
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

# Define a search tool
@tool
def search_web(query: str) -> str:
    """Search the web for information.
    
    Args:
        query: What to search for
    
    Returns:
        Search results
    """
    return f"[Results for '{query}'] Found 10 articles."

tools = [calculator, search_web]
print("✅ Tools created:", [t.name for t in tools])

**We just created tools!**
- `@tool` decorator makes functions callable by the agent
- `calculator` - does math properly
- `search_web` - searches the web

Now let's bind them to an LLM.

---

## 🔗 Step 4: Creating an Agent

In [ ]:
from langchain_core.messages import HumanMessage

# Create LLM with tools
llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(tools)

# Test: Ask a math question
response = llm_with_tools.invoke([
    HumanMessage(content="What is 25 * 17?")
])

print("🔍 Response type:", type(response).__name__)
print("🔧 Tool calls:", response.tool_calls)
print("💬 Content:", response.content)

**See what happened!**
- We asked about math
- The LLM saw it has a `calculator` tool
- It decided to CALL the tool (look at `tool_calls`!)
- It didn't just answer - it took ACTION!

In [ ]:
# Now run the tool!
for tool_call in response.tool_calls:
    if tool_call["name"] == "calculator":
        result = calculator.invoke(tool_call["arguments"])
        print(f"📊 Calculator result: {result}")

---

## 🔄 Step 5: The Agent Loop

In [ ]:
# Full agent with loop
def run_agent(query, max_steps=5):
    """Run agent with tools."""
    from langchain_core.messages import HumanMessage, AIMessage
    
    messages = [HumanMessage(content=query)]
    
    for step in range(max_steps):
        print(f"\n📍 Step {step + 1}:")
        
        # Get response
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        
        # Check if tools called
        if response.tool_calls:
            print(f"   🔧 Calling: {[tc['name'] for tc in response.tool_calls]}")
            
            # Execute tools
            for tc in response.tool_calls:
                tool_name = tc["name"]
                if tool_name == "calculator":
                    result = calculator.invoke(tc["arguments"])
                elif tool_name == "search_web":
                    result = search_web.invoke(tc["arguments"])
                else:
                    result = "Unknown tool"
                print(f"   📦 Result: {result}")
                messages.append(Human_msg := HumanMessage(content=str(result)))
        else:
            print(f"   ✅ Done: {response.content}")
            return response.content
    
    return "Max steps reached"

# Test the agent
result = run_agent("What's the weather in Paris?")

**Wow! That's the agent loop in action:**
1. User asks question
2. LLM decides to call a tool
3. Tool executes
4. Result goes back to LLM
5. LLM forms final answer

This is exactly how ChatGPT, Claude, etc. work!

---

## 💾 Step 6: Memory

In [ ]:
# Simple conversation memory
conversation_history = []

# Turn 1
conversation_history.append(HumanMessage(content="My name is Gemma."))
response1 = llm.invoke(conversation_history)
conversation_history.append(response1)
print("Turn 1 - I said: My name is Gemma")
print(f"        AI: {response1.content[:50]}...")

# Turn 2 - with memory!
conversation_history.append(HumanMessage(content="What's my name?"))
response2 = llm.invoke(conversation_history)
print("\nTurn 2 - I asked: What's my name?")
print(f"         AI: {response2.content}")

**Memory works!**
- We pass the FULL conversation each time
- The LLM sees the context
- It knows who you are!

---

## 📊 Step 7: Evaluating the Agent

In [ ]:
# Simple evaluation
test_cases = [
    ("What is 2+2?", "4"),
    ("Capital of France?", "Paris"),
    ("What's 10 * 10?", "100"),
]

print("📊 Running Evaluation...")
print("=" * 50)

passed = 0
for query, expected in test_cases:
    result = llm.invoke([HumanMessage(content=query)])
    answer = result.content
    
    # Check if expected is in answer
    is_correct = expected.lower() in answer.lower()
    status = "✅" if is_correct else "❌"
    
    print(f"{status} Query: {query}")
    print(f"   Expected: {expected} | Got: {answer[:30]}...")
    
    if is_correct:
        passed += 1
    print()

print(f"📈 Score: {passed}/{len(test_cases)} passed ({100*passed/len(test_cases):.0f}%)")

**That's eval basics!**
- Test cases = queries + expected answers
- Run agent on each
- Check if output contains expected

Real agent evals are more sophisticated: check tools used, trajectory, no hallucinations!

---

## 🎉 What We Built!

In [ ]:
print("""
┌─────────────────────────────────────────────────────┐
│         WHAT WE BUILT TODAY                        │
├─────────────────────────────────────────────────────┤
│  ✅ LLM - Basic language model                  │
│  ✅ Tools - Calculator, search                 │
│  ✅ Agent - With tool-calling loop            │
│  ✅ Memory - Conversation history          │
│  ✅ Evaluation - Test cases + scoring      │
├─────────────────────────────────────────────────────┤
│  Next steps:                                │
│  • More tools (email, calendar, etc.)       │
│  • LangGraph for complex agents           │
│  • LangSmith for full tracing            │
│  • Production evals with datasets        │
└─────────────────────────────────────────────────────┘
print("✅ Tutorial complete!")

---

## 📚 Resources
- LangChain: https://python.langchain.com
- LangSmith: https://smith.langchain.com
- LangGraph: https://langchain-ai.github.io/langgraph/

**Go forth and build agents!** 🤖